In [3]:
import pandas as pd

# Membaca dataset
df_raw = pd.read_csv('ai_student_impact_dataset (1).csv')

# Transfrom (Data Cleaning)
df_cleaned = df_raw.drop_duplicates().dropna()

# -- TABEL DIMENSI ---

# Dimensi Student
dim_student = df_cleaned[['Student_ID','Major_Category', 'Year_of_Study']].drop_duplicates().reset_index(drop=True)

# Dimensi AI Profile
dim_ai_profile = df_cleaned[['Primary_Use_Case','Prompt_Engineering_Skill','Paid_Subscription']].drop_duplicates().reset_index(drop=True)
dim_ai_profile['ai_profile_id'] =  dim_ai_profile.index
dim_ai_profile = dim_ai_profile[['ai_profile_id', 'Primary_Use_Case', 'Prompt_Engineering_Skill', 'Paid_Subscription']]

# Dimensi Kebijakan Kampus
dim_policy = df_cleaned[['Institutional_Policy']].drop_duplicates().reset_index(drop=True)
dim_policy['policy_id'] = dim_policy.index
dim_policy = dim_policy[['policy_id', 'Institutional_Policy']]

# Dimensi Burnout Resiko
dim_risk = df_cleaned[['Burnout_Risk_Level']].drop_duplicates().reset_index(drop=True)
dim_risk['risk_id'] = dim_risk.index
dim_risk = dim_risk[['risk_id', 'Burnout_Risk_Level']]

# --- TABEL FAKTA ---
fact_table = df_cleaned.merge(dim_student, on=['Student_ID','Major_Category', 'Year_of_Study'])\
.merge(dim_ai_profile, on=['Primary_Use_Case','Prompt_Engineering_Skill','Paid_Subscription'])\
.merge(dim_policy, on=['Institutional_Policy'])\
.merge(dim_risk, on=['Burnout_Risk_Level'])

fact_table = fact_table[[
    'Student_ID', 'ai_profile_id', 'policy_id', 'risk_id',
    'Pre_Semester_GPA', 'Post_Semester_GPA', 'Weekly_GenAI_Hours', 
    'Traditional_Study_Hours', 'Tool_Diversity', 'Perceived_AI_Dependency', 
    'Anxiety_Level_During_Exams', 'Skill_Retention_Score'
]].reset_index(drop=True)

# Primary Key
fact_table['fact_id'] = fact_table.index
fact_table = fact_table[['fact_id'] + [c for c in fact_table.columns if c != 'fact_id']]


# --- PROSES LOAD KE SUPABASE
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")

engine = create_engine(DATABASE_URL)

print("Koneksi berhasil")

# Kirim data dimensi ke Supabase
dim_student.to_sql('dim_student', con=engine, if_exists='replace', index=False)
dim_ai_profile.to_sql('dim_ai_profile', con=engine, if_exists='replace', index=False)
dim_policy.to_sql('dim_policy', con=engine, if_exists='replace', index=False)
dim_risk.to_sql('dim_risk', con=engine, if_exists='replace', index=False)
print("Semua tabel dimensi berhasil mendarat!")

print("Memulai proses LOAD data fakta...")
# Kirim data fakta ke Supabase (Sesuaikan nama variabel fact_table di jupitermu)
fact_table.to_sql('fact_student_ai_impact', con=engine, if_exists='append', index=False)
print("FASE LOAD SELESAI 100%! Data fakta berhasil di-input.")


Koneksi berhasil
Semua tabel dimensi berhasil mendarat!
Memulai proses LOAD data fakta...
FASE LOAD SELESAI 100%! Data fakta berhasil di-input.
